# Stage A — Qwen3-4B weight experiment (offline)

- 5 cấu hình × seed 42
- 5M token/run, 625 optimizer step
- chỉ train + generate; COMET/Gemma chấm sau ở môi trường có internet
- giữ nguyên dataset/competition có sẵn của notebook


In [ ]:
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path

import torch
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('Không có GPU; dừng để tiết kiệm quota')
p = torch.cuda.get_device_properties(0)
vram = p.total_memory / 1e9
print('gpu', p.name, round(vram, 1), 'GB | capability', torch.cuda.get_device_capability(0))
if vram < 40:
    raise SystemExit(f'GPU {p.name} chỉ {vram:.1f} GB, không phải RTX Pro 6000; dừng trước khi train')


In [ ]:
# Cài wheel offline (không internet), chỉ 6 package; bỏ qua nếu lỗi.
WHEELS = Path('/kaggle/input/tiki-offline-wheels')
if WHEELS.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    '--find-links', str(WHEELS), '--no-deps',
                    'transformers==5.22.0', 'peft==0.21.2',
                    'accelerate==1.15.0', 'huggingface_hub==1.33.0',
                    'tokenizers==0.23.3', 'safetensors==0.8.0'], check=False)
import transformers, peft, accelerate
print('transformers', transformers.__version__, '| peft', peft.__version__,
      '| accelerate', accelerate.__version__)


In [ ]:
# Tìm dataset (Kaggle có thể mount folder hoặc file zip).
DATA_ROOT = Path('/kaggle/input/tiki-4model-eval')
if not (DATA_ROOT / 'data/weight_eval/eval_1688.parquet').exists():
    zips = sorted(DATA_ROOT.glob('*.zip'))
    if zips:
        dest = Path('/kaggle/working/tiki-4model-eval')
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(dest)
        DATA_ROOT = dest
print('dataset', DATA_ROOT)
assert (DATA_ROOT / 'data/weight_eval/eval_1688.parquet').exists()
print('eval rows', len(__import__('pandas').read_parquet(DATA_ROOT/'data/weight_eval/eval_1688.parquet')))


In [ ]:
# Tìm Qwen3-4B đã attach làm model source (không tải mạng).
MODEL = None
for cfg_path in glob.glob('/kaggle/input/**/config.json', recursive=True):
    try:
        c = json.loads(Path(cfg_path).read_text(encoding='utf-8'))
    except Exception:
        continue
    if (c.get('hidden_size'), c.get('intermediate_size'),
            c.get('num_hidden_layers')) == (2560, 9728, 36):
        MODEL = Path(cfg_path).parent
        break
print('model', MODEL)
if MODEL is None:
    raise SystemExit('Không thấy Qwen3-4B trong /kaggle/input; kiểm tra model_sources')
print(sorted(x.name for x in MODEL.glob('*'))[:20])


In [ ]:
# Smoke 2 step: chỉ kiểm tra load + backward + generate, không đụng eval set.
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
tok = AutoTokenizer.from_pretrained(str(MODEL), trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
try:
    base = AutoModelForCausalLM.from_pretrained(str(MODEL), dtype=torch.bfloat16,
                                                trust_remote_code=True, low_cpu_mem_usage=True)
except TypeError:
    base = AutoModelForCausalLM.from_pretrained(str(MODEL), torch_dtype=torch.bfloat16,
                                                trust_remote_code=True, low_cpu_mem_usage=True)
base.gradient_checkpointing_enable(); base.enable_input_require_grads()
smoke = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                                        task_type='CAUSAL_LM', target_modules='all-linear')).to('cuda')
smoke.train()
batch = tok(['Áo thun nam cotton', 'Bình nước 2 lít'], return_tensors='pt', padding=True).to('cuda')
batch['labels'] = batch['input_ids'].clone()
opt = torch.optim.AdamW([q for q in smoke.parameters() if q.requires_grad], lr=1e-4)
for i in range(2):
    loss = smoke(**batch).loss
    loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    print('smoke step', i, 'loss', round(float(loss.item()), 4))
with torch.inference_mode():
    gen = smoke.generate(batch['input_ids'][:1], max_new_tokens=4, do_sample=False)
print('smoke generate ok', gen.shape)
del smoke, base, opt; torch.cuda.empty_cache()
print('SMOKE OK')


## Stage A: 5 cấu hình × seed 42


In [ ]:
OUT = Path('/kaggle/working/weight_runs')
OUT.mkdir(parents=True, exist_ok=True)
CONFIGS = ['natural', 'balanced', 'no_desc', 'tiki_heavy', 'title_heavy']
MB = 8 if vram >= 60 else (4 if vram >= 30 else 2)
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
           TOKENIZERS_PARALLELISM='false')
started = time.time()
for cfg in CONFIGS:
    hyp = OUT / f'hyp_{cfg}_seed42.parquet'
    if hyp.exists():
        print('skip existing', hyp, flush=True); continue
    cmd = [sys.executable, str(DATA_ROOT / 'code/train_weight_gpu.py'),
           '--model', str(MODEL),
           '--train-view', str(DATA_ROOT / 'data/weight_eval/train_view_4b.parquet'),
           '--eval-dir', str(DATA_ROOT / 'data/weight_eval'),
           '--configs', cfg, '--seeds', '42',
           '--tokens', '5000000', '--steps', '625',
           '--out', str(OUT), '--dtype', 'bf16', '--micro-batch', str(MB)]
    print('RUN', cfg, 'micro-batch', MB, flush=True)
    subprocess.run(cmd, check=True, env=env)
print('STAGE A DONE in', round((time.time()-started)/60, 1), 'min')


In [ ]:
for p in sorted(OUT.glob('hyp_*.parquet')):
    print(p.name, p.stat().st_size)
print('--- adapters ---')
for p in sorted(OUT.glob('adapter_*')):
    print(p.name)
(OUT / 'stage_a_done.json').write_text(json.dumps({'configs': CONFIGS, 'seed': 42,
    'vram_gb': round(vram, 1), 'gpu': torch.cuda.get_device_name(0),
    'finished_at': time.time()}), encoding='utf-8')
